****IMPORTING REQUIRED LIABRARIES****

In [1]:
#import required liabraries
import numpy as np  
import pandas as pd
from datetime import datetime
from dateutil import parser

#Load the dataset
df = pd.read_csv('Week-3-Employee-Data (2).csv')
df.head()

,employee_id,first_name,last_name,email,gender,department,job_title,hire_date,exit_date,is_active,salary,currency,country,state,city,manager_id,performance_score
0,1001,Ken,Naidoo,ken.naidoo@example.com,Male,HR,Sales Rep,29/05/2017,NaN,True,"55,286.31",GBP,south-africa,Western Cape,NaN,NaN,NaN
1,1002,Sam,Smith,sam.smith@mail.com,M,H.R.,Sales Rep,04-27-2016,NaN,True,29408.75,GBP,south-africa,NaN,Johannesburg,1001.0,C
2,1003,Oscar,Mthembu,oscar.mthembu@mail.com,Male,Ops,Engineer,12-02-2015,NaN,False,"35,393.84",$,SA,KZN,Cape Town,NaN,NaN
3,1004,Sam,Brown,sam.brown@example,M,Sales,Senior Analyst,01-19-2016,NaN,True,39890.2,£,USA,KZN,NaN,1003.0,C
4,1005,Faith,Mthembu,faith.mthembu@mail.com,M,H.R.,Engineer,06-18-2019,2022/01/22,NaN,25532.92,GBP,United Kingdom,NaN,Polokwane,NaN,B


****HANDLING DATE COLUMNS****

In [2]:
# Step 1: clean obvious junk
df['hire_date'] = df['hire_date'].replace(['', ' ', 'N/A', 'na'], pd.NA)
df['exit_date'] = df['exit_date'].replace(['', ' ', 'N/A', 'na'], pd.NA)

# Step 2: robust mixed-format parser
def parse_mixed_date(x):
    try:
        return parser.parse(str(x), dayfirst=True)
    except:
        return pd.NaT

df['hire_date'] = df['hire_date'].apply(parse_mixed_date)
df['exit_date'] = df['exit_date'].apply(parse_mixed_date)

# Step 3: business rule
df.loc[df['is_active'] == True, 'exit_date'] = pd.NaT

# Step 4: validate
df[['hire_date', 'exit_date']].isna().sum()


hire_date      0
exit_date    106
dtype: int64

***Data types: salary and manager_id to numeric.***

In [3]:
##salary and manager_id to numeric.
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df['manager_id'] = pd.to_numeric(df['manager_id'], errors='coerce')


****Whitespace & casing: trim strings; title-case names; lower-case emails.****

In [4]:
# --- Whitespace & Casing ---
str_cols = df.select_dtypes(include='object').columns

for col in str_cols:
    df[col] = df[col].astype(str).str.strip()



df['first_name'] = df['first_name'].str.title()
df['last_name'] = df['last_name'].str.title()
df['email'] = df['email'].str.lower()

****Standardizing categories****

In [5]:
# --- Standardizing Departments ---

# 1. Convert to lowercase & strip whitespace first
df['department'] = df['department'].astype(str).str.strip().str.lower()

# 2. Mapping all known variants
dept_map = {
    # Human Resources
    'hr': 'Human Resources',
    'h.r.': 'Human Resources',
    'human resources': 'Human Resources',
    'human resource': 'Human Resources',

    # Information Technology
    'IT':'Information Technology',
    'IT': 'information technology',
    'it': 'Information Technology',

    # Finance
    'finance': 'Finance',
    'fin': 'Finance',
    'financial': 'Finance',

    # Sales
    'sales': 'Sales',
    'sales dept': 'Sales',
    'sales department': 'Sales',

    #  #Engineering
    'eng': 'Engineering',
    'engineering': 'Engineering',


    # Operations
    'operations': 'Operations',
    'ops': 'Operations',
    'operation': 'Operations',

}


# 3. Applying mapping
df['department'] = df['department'].replace(dept_map)

# 4. Replacing 'nan' strings back to NaN
df['department'] = df['department'].replace('nan', np.nan)

# 5. Filling the missing departments with mode (most common)
df['department'] = df['department'].fillna(df['department'].mode()[0])

df['department'].value_counts()

department
Human Resources           40
Engineering               28
Sales                     22
Finance                   19
Operations                17
information technology    15
Information Technology    14
Name: count, dtype: int64

****Country mapping****

In [6]:
#country mapping
country_mapping = {
    'USA': 'United States',
    'US': 'United States',
    'United States': 'United States',
    'UK': 'United Kingdom',
    'United Kingdom': 'United Kingdom',
    'GB': 'United Kingdom',
    'SA': 'South Africa',
    'South Africa': 'South Africa',
    'ZA': 'South Africa',
    'RSA': 'South Africa',
    'U.K.' : 'United Kingdom',
    'south-africa' : 'South Africa'
}
 
df['country'] = df['country'].str.strip().replace(country_mapping)
df.head()

,employee_id,first_name,last_name,email,gender,department,job_title,hire_date,exit_date,is_active,salary,currency,country,state,city,manager_id,performance_score
0,1001,Ken,Naidoo,ken.naidoo@example.com,Male,Human Resources,Sales Rep,2017-05-29,NaT,True,NaN,GBP,South Africa,Western Cape,nan,NaN,nan
1,1002,Sam,Smith,sam.smith@mail.com,M,Human Resources,Sales Rep,2016-04-27,NaT,True,29408.75,GBP,South Africa,nan,Johannesburg,1001.0,C
2,1003,Oscar,Mthembu,oscar.mthembu@mail.com,Male,Operations,Engineer,2015-02-12,NaT,False,NaN,$,South Africa,KZN,Cape Town,NaN,nan
3,1004,Sam,Brown,sam.brown@example,M,Sales,Senior Analyst,2016-01-19,NaT,True,39890.20,£,United States,KZN,nan,1003.0,C
4,1005,Faith,Mthembu,faith.mthembu@mail.com,M,Human Resources,Engineer,2019-06-18,2022-01-22,nan,25532.92,GBP,United Kingdom,nan,Polokwane,NaN,B


In [7]:

# --- Normalizing Currency Codes ---
currency_map = {
    'R': 'ZAR',
    '£': 'GBP',
    '$': 'USD'
}

df['currency'] = df['currency'].replace(currency_map)

In [8]:
# --- Removing Duplicates ---
df = df.drop_duplicates(subset='employee_id')
df.duplicated(subset='employee_id').sum()

np.int64(0)

In [9]:
# --- Basic Validations ---
# Invalid emails
df['invalid_email'] = ~df['email'].str.match(r'.+@.+\..+', na=False)

# Negative salaries -> set to NaN
df.loc[df['salary'] < 0, 'salary'] = np.nan

# Exit date earlier than hire date
df['invalid_exit_date'] = df['exit_date'] < df['hire_date']

In [10]:
# Saving cleaned file
df.to_csv('Week-3-Employee-Data_CLEANED.csv', index=False)
df.head()

,employee_id,first_name,last_name,email,gender,department,job_title,hire_date,exit_date,is_active,salary,currency,country,state,city,manager_id,performance_score,invalid_email,invalid_exit_date
0,1001,Ken,Naidoo,ken.naidoo@example.com,Male,Human Resources,Sales Rep,2017-05-29,NaT,True,NaN,GBP,South Africa,Western Cape,nan,NaN,nan,False,False
1,1002,Sam,Smith,sam.smith@mail.com,M,Human Resources,Sales Rep,2016-04-27,NaT,True,29408.75,GBP,South Africa,nan,Johannesburg,1001.0,C,False,False
2,1003,Oscar,Mthembu,oscar.mthembu@mail.com,Male,Operations,Engineer,2015-02-12,NaT,False,NaN,USD,South Africa,KZN,Cape Town,NaN,nan,False,False
3,1004,Sam,Brown,sam.brown@example,M,Sales,Senior Analyst,2016-01-19,NaT,True,39890.20,GBP,United States,KZN,nan,1003.0,C,True,False
4,1005,Faith,Mthembu,faith.mthembu@mail.com,M,Human Resources,Engineer,2019-06-18,2022-01-22,nan,25532.92,GBP,United Kingdom,nan,Polokwane,NaN,B,False,False


****CALCULATING KPI'S****

****Active Employees (current headcount)****


In [11]:

# a) Current Headcount
df['is_active_clean'] = (
    df['is_active']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'true': True,
        '1': True,
        'yes': True,
        'false': False,
        '0': False,
        'no': False
    })
)

current_headcount = df['is_active_clean'].sum()

print(f"Current Headcount (Active Employees): {current_headcount}")


Current Headcount (Active Employees): 52


****Hires by Year****

In [12]:
# b) Hires by Year
df['hire_year'] = df['hire_date'].dt.year
hires_by_year = df['hire_year'].value_counts().sort_index()
hires_by_year

hire_year
2015    18
2016    18
2017    27
2018    20
2019    18
2020    13
2021    13
2022    23
Name: count, dtype: int64

****Average Salary by Department and Currency****

In [13]:
# c) Average Salary by Department and Currency
avg_salary = df.groupby(['department', 'currency'])['salary'].mean().round(2)
avg_salary


department              currency
Engineering             GBP         45461.64
                        USD         29981.22
                        ZAR         32755.47
Finance                 GBP         35511.43
                        USD         95672.50
                        ZAR         39023.12
Human Resources         GBP         32719.79
                        USD         54174.53
                        ZAR         34874.18
Information Technology  GBP         27850.84
                        USD         37954.08
                        ZAR         36220.15
Operations              GBP         28019.71
                        USD         35270.06
                        ZAR         31503.66
Sales                   GBP         25249.97
                        USD         35982.63
                        ZAR         32468.51
information technology  GBP         33485.78
                        USD         35278.03
                        ZAR         35861.11
Name: salary, dtype: f

****Median Tenure (days) for active employees****

In [14]:

# d) Median Tenure (days) for active employees
df['is_active'] = (
    df['is_active']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({'true': True, 'false': False})
)


df['hire_date'] = pd.to_datetime(df['hire_date'], errors='coerce', dayfirst=True)
df['exit_date'] = pd.to_datetime(df['exit_date'], errors='coerce', dayfirst=True)

today = pd.Timestamp.today().normalize() # Use today's date

# Calculating tenure in days
df['tenure_days'] = np.where(
    df['exit_date'].notna(),
    (df['exit_date'] - df['hire_date']).dt.days,
    (today - df['hire_date']).dt.days
)
df[
    (df['is_active'] == True)
][['hire_date', 'exit_date', 'tenure_days']].head(10)

#Calculating median tenure for ACTIVE employees only
median_tenure = int(
    df[
        (df['is_active'] == True) &
        (df['hire_date'].notna()) &
        (df['tenure_days'].notna())
    ]['tenure_days']
    .median()
)

print(f"Median Tenure (days) for active employees: {median_tenure}")



Median Tenure (days) for active employees: 2535


****Turnover rate****

In [15]:
# e) Turnover Rate
turnover_rate = (df['exit_date'].notna().sum() / len(df)) * 100
turnover_rate

np.float64(31.333333333333336)